In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df=pd.read_csv("/content/email.csv")

In [ ]:
df.head(5)

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5573 non-null   object
 1   Message   5573 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [ ]:
df['Category'].value_counts()

,count
Category,
ham,4825
spam,747
"{""mode"":""full""",1


In [ ]:
df.isnull().sum()

,0
Category,0
Message,0


In [ ]:
df.duplicated().sum()

np.int64(415)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(5158, 2)

In [ ]:
df['Category'] = df['Category'].map({
    'ham': 0,
    'spam': 1
})

In [ ]:
print(df['Category'])

0       0.0
1       0.0
2       1.0
3       0.0
4       0.0
       ... 
5568    0.0
5569    0.0
5570    0.0
5571    0.0
5572    NaN
Name: Category, Length: 5158, dtype: float64


In [ ]:
df['Category'].value_counts()

,count
Category,
0.0,4516
1.0,641


In [ ]:
df['Message']=df['Message'].str.lower()

In [ ]:
import re
df['Message']=df['Message'].apply(lambda text:re.sub(r'[^\w\s]','',text))

In [ ]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

df['Message'] = df['Message'].apply(word_tokenize)

print(df[['Message']].head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


                                             Message
0  [go, until, jurong, point, crazy, available, o...
1                     [ok, lar, joking, wif, u, oni]
2  [free, entry, in, 2, a, wkly, comp, to, win, f...
3  [u, dun, say, so, early, hor, u, c, already, t...
4  [nah, i, dont, think, he, goes, to, usf, he, l...


In [ ]:
print(df['Message'])

0       [go, until, jurong, point, crazy, available, o...
1                          [ok, lar, joking, wif, u, oni]
2       [free, entry, in, 2, a, wkly, comp, to, win, f...
3       [u, dun, say, so, early, hor, u, c, already, t...
4       [nah, i, dont, think, he, goes, to, usf, he, l...
                              ...                        
5568         [will, ü, b, going, to, esplanade, fr, home]
5569    [pity, was, in, mood, for, that, soany, other,...
5570    [the, guy, did, some, bitching, but, i, acted,...
5571                     [rofl, its, true, to, its, name]
5572                                      [isactivefalse]
Name: Message, Length: 5158, dtype: object


In [ ]:
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

df['Message'] = df['Message'].apply(
    lambda words: [
        word for word in words
        if word not in stop_words
    ]
)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df.head()

,Category,Message
0,0.0,"[go, jurong, point, crazy, available, bugis, n..."
1,0.0,"[ok, lar, joking, wif, u, oni]"
2,1.0,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,0.0,"[u, dun, say, early, hor, u, c, already, say]"
4,0.0,"[nah, dont, think, goes, usf, lives, around, t..."


In [ ]:
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()
df['Message']=df['Message'].apply(
    lambda words:' '.join(
        lemmatizer.lemmatize(word)
        for word in words
    )
)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
df.head()

,Category,Message
0,0.0,go jurong point crazy available bugis n great ...
1,0.0,ok lar joking wif u oni
2,1.0,free entry 2 wkly comp win fa cup final tkts 2...
3,0.0,u dun say early hor u c already say
4,0.0,nah dont think go usf life around though


In [ ]:
df = df.dropna(subset=['Category'])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
x=tfidf.fit_transform(df['Message'])
y=df['Category']

In [ ]:
df.head()

,Category,Message
0,0.0,go jurong point crazy available bugis n great ...
1,0.0,ok lar joking wif u oni
2,1.0,free entry 2 wkly comp win fa cup final tkts 2...
3,0.0,u dun say early hor u c already say
4,0.0,nah dont think go usf life around though


In [ ]:
print(x.shape)

(5157, 8885)


In [ ]:
df['Category'].value_counts()

,count
Category,
0.0,4516
1.0,641


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

model.fit(x_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test,y_pred))


0.9437984496124031


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.94      1.00      0.97       896
         1.0       0.95      0.60      0.74       136

    accuracy                           0.94      1032
   macro avg       0.95      0.80      0.85      1032
weighted avg       0.94      0.94      0.94      1032



In [ ]:
message = ["Hey bro, where are you?"]

message_tfidf = tfidf.transform(message)

prediction = model.predict(message_tfidf)

if prediction[0] == 1:
    print("Spam")
else:
    print("Ham")

Ham


In [ ]:
message = ["Congratulations! You have won ₹50,000 cash prize. Click the link below to claim your reward now."]

message_tfidf = tfidf.transform(message)

prediction = model.predict(message_tfidf)

if prediction[0] == 1:
    print("Spam")
else:
    print("Ham")

Spam


In [ ]:
print(message_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7 stored elements and shape (1, 8885)>
  Coords	Values
  (0, 671)	0.4179296633916624
  (0, 1972)	0.31269051988719004
  (0, 2140)	0.28992705203905306
  (0, 2160)	0.4385025061058332
  (0, 4764)	0.4074134418990501
  (0, 6236)	0.3048723197729364
  (0, 6615)	0.4385025061058332
